In [42]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [43]:
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DATA_PATH = os.getenv(
    "DATA_PATH"
)

df_comments=pd.read_csv(f'{DATA_PATH}df_n4.csv')
df_context=pd.read_csv(f'{DATA_PATH}df_n2.csv')


In [44]:

import ast

df = df_comments.merge(
    df_context[
        [
            "patch_id",
            "relevant_context",
            "hunk",
            "target_file",
            "pr_title",
            "changed_files",
            "relevant_same_file_code_hunks"
        ]
    ],
    on="patch_id",
    how="left",
)

def get_first_comment(comments):
    """
    After deduplication, every cluster should have one comment representing it:
    duplicate cluster → use the synthesized comment generated by the LLM;
    singleton cluster → there is nothing to synthesize, so just use the original comment.
    """
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except Exception:
            return comments

    if isinstance(comments, list):
        return comments[0] if len(comments) > 0 else None

    return comments

df["comment"] = df.apply(
    lambda row: (
        row["synthesis_comment"]
        if pd.notna(row["synthesis_comment"])
        else get_first_comment(row["comments"])
    ),
    axis=1,
)

df=df.drop(columns=['comments','synthesis_comment'])


In [45]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

# ==========================
# OpenAI Configuration
# ==========================

config_str = os.getenv("GPT_LUNA_CONFIG")

if config_str is None:
    raise ValueError("model environment variable is missing")

config = json.loads(config_str)
MODEL_NAME = config["name"]
REASONING_EFFORT = config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

SLEEP_BETWEEN_CALLS = int(
    os.getenv("SLEEP_BETWEEN_CALLS", "1")
)


client = OpenAI(
    api_key=OPENAI_API_KEY
)


Using model: gpt-5.6-luna with reasoning effort: low


In [46]:
import json
import time
import random
import pandas as pd

TOTAL_USAGE = {
    "num_calls": 0,
    "prompt_tokens": 0,
    "reasoning_tokens": 0,
    "total_tokens": 0,
}


def update_usage(usage):
    if usage is None:
        return
    TOTAL_USAGE["num_calls"] += 1
    TOTAL_USAGE["prompt_tokens"] += getattr(usage, "prompt_tokens", 0) or 0
    TOTAL_USAGE["total_tokens"] += getattr(usage, "total_tokens", 0) or 0

    reasoning_tokens = 0
    details = getattr(usage, "completion_tokens_details", None)
    if details is not None:
        reasoning_tokens = getattr(details, "reasoning_tokens", 0) or 0
    TOTAL_USAGE["reasoning_tokens"] += reasoning_tokens


# -----------------------------
# PROMPT BUILDING
# -----------------------------
def build_messages(row):

    system_prompt = """
You are a highly skilled software engineer with extensive experience reviewing pull requests.

Your task is to evaluate the relevance of a code review comment for a given code change.

A highly relevant review comment:
- identifies a real issue or meaningful improvement related to the code change,
- is supported by the provided patch and surrounding context,
- is specific and actionable,
- is concise without omitting important information,
- does not discuss unrelated or pre-existing issues.

A low-relevance review comment:
- discusses code unrelated to the patch,
- makes unsupported assumptions,
- is factually incorrect,
- is too vague or generic,
- or provides little useful value.

Evaluate the review comment using ONLY the provided pull request information, code patch, and surrounding context. Do not assume code or project behavior that is not shown.

Assign a relevance score from 1 to 5 using the following rubric:

1 = Completely irrelevant
2 = Mostly irrelevant
3 = Partially relevant
4 = Mostly relevant
5 = Highly relevant

Return ONLY valid JSON in the following format:

{
  "score": <integer between 1 and 5>,
  "reason": "<short explanation>"
}

Do not return any text outside the JSON object.
"""

    user_prompt = f"""
PULL REQUEST TITLE:
{row['pr_title']}

TARGET FILE:
{row['target_file']}

RELATED CODE HUNKS IN THE SAME FILE:
<related_hunks>
{row['relevant_same_file_code_hunks']}
</related_hunks>

SURROUNDING CONTEXT:
<context>
{row['relevant_context']}
</context>

CODE PATCH:
<patch>
{row['hunk']}
</patch>

REVIEW COMMENT:
{row['comment']}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def parse_output(text):

    if text is None:
        return None, None

    text = str(text).strip()

    try:
        data = json.loads(text)

        score = data.get("score")

        try:
            score = int(score)
        except Exception:
            score = None

        if score not in [1, 2, 3, 4, 5]:
            score = None

        reason = str(data.get("reason", "")).strip()

        return score, reason

    except Exception:
        return None, text


# ---------------------------------------------------
# ONE PREDICTION
# ---------------------------------------------------

def predict_once(row):

    messages = build_messages(row)

    kwargs = {
        "model": MODEL_NAME,
        "messages": messages,
        "timeout": 60,
    }

    if REASONING_EFFORT:
        kwargs["reasoning_effort"] = REASONING_EFFORT

    response = client.chat.completions.create(**kwargs)

    raw_text = response.choices[0].message.content

    score, reason = parse_output(raw_text)

    update_usage(response.usage)

    return {
        "Rel_Pred": score,
        "reason": reason,
    }


# ---------------------------------------------------
# PIPELINE
# ---------------------------------------------------

def run_pipeline(df):

    df = df.copy()

    predictions = []

    total = len(df)

    for idx, row in df.iterrows():

        print(f"{idx+1}/{total}")

        if (
            pd.isna(row["hunk"])
            or row["comment"] is None
            or row["comment"] == ""
        ):

            predictions.append(
                {
                    "Rel_Pred": None,
                    "reason": "missing input",
                }
            )

            continue

        success = False
        sleep_time = SLEEP_BETWEEN_CALLS

        for attempt in range(3):

            try:

                pred = predict_once(row)

                predictions.append(pred)

                success = True

                break

            except Exception as e:

                print(
                    f"Row {idx+1} | Attempt {attempt+1}/3 | {e}"
                )

                wait = sleep_time + random.uniform(0, 1)

                print(f"Retrying in {wait:.2f}s")

                time.sleep(wait)

                sleep_time *= 2

        if not success:

            predictions.append(
                {
                    "Rel_Pred": None,
                    "reason": "API_ERROR",
                }
            )

        time.sleep(SLEEP_BETWEEN_CALLS + random.uniform(0, 0.8))

        # Backup every 5 predictions
        if (idx + 1) % 5 == 0:

            backup = pd.concat(
                [
                    df.iloc[: idx + 1].reset_index(drop=True),
                    pd.DataFrame(predictions),
                ],
                axis=1,
            )

            backup.to_csv(
                "artifacts/backup_predictions.csv",
                index=False,
            )

    pred_df = pd.DataFrame(predictions)

    final_df = pd.concat(
        [
            df.reset_index(drop=True),
            pred_df,
        ],
        axis=1,
    )
    print(
        "[DONE] Finished pipeline"
    )

    columns = [
        "patch_id",
        "cluster_id",
        "num_comments",
        "generation_systems",
        "categories",
        "severities",
        "comment",
        "Rel_Pred",
        "reason",
    ]

    return final_df[columns]

In [ ]:
results=run_pipeline(df)
results.to_csv(f'{DATA_PATH}df_n5.csv', index=False)

1/1
[DONE] Finished pipeline


In [48]:
results['Rel_Pred'].value_counts()


Rel_Pred
3    1
Name: count, dtype: int64

In [49]:
import json
from datetime import datetime
from pathlib import Path


def save_token_usage_log(
    task_name="relevance_judgement",
    log_file="../logs/token_usage_insights_logs.json"
):

    usage_stats = {
        "task": task_name,
        "timestamp": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }

    log_path = Path(log_file)

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with open(log_path, "r") as f:
            logs = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        logs = []

    logs.append(usage_stats)

    with open(log_path, "w") as f:
        json.dump(
            logs,
            f,
            indent=4
        )

    print(f"Token usage saved to {log_path}")
    
save_token_usage_log()    

Token usage saved to ..\logs\token_usage_insights_logs.json
